# 01 — NumPy approfondi

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- créer et manipuler des `ndarray` avec leurs `dtype` adaptés;
- comprendre et exploiter le **broadcasting** et la vectorisation;
- utiliser les **ufuncs** et les fonctions universelles;
- pratiquer l'indexation avancée (boolean masks, fancy indexing, slicing);
- faire un peu d'algèbre linéaire avec `numpy.linalg`;
- générer des nombres aléatoires reproductibles avec `numpy.random.Generator`;


## Prérequis — ce que vous devez maîtriser

Ce notebook fait partie des **Side tracks**. Il est conçu pour être autonome, à condition d'avoir suivi le niveau **Intermédiaire** de la formation principale. Vous devez donc être à l'aise avec :

- les types de base Python (`int`, `float`, `list`, `tuple`);
- les fonctions typées et les `list`/`dict` comprehensions;
- les modules standards `math`, `statistics`;
- l'utilisation de Jupyter (cellules, affichage);

Ce que ce notebook **ne couvrira pas** (ou seulement en mention) :

- les concepts de machine learning (voir `07_Scikit_learn_intro.ipynb`);
- la visualisation (voir `Data_visualization/01_Matplotlib_OO_complet.ipynb`);
- Pandas (voir `02_Pandas_approfondi.ipynb`);


> **Note.** NumPy est la brique de calcul scientifique de Python. Tous les autres modules data (Pandas, Polars, SciPy, scikit-learn, PyTorch) reposent dessus ou peuvent interopérer avec ses `ndarray`.


## Plan

1. Installation et import
2. Créer un `ndarray`
3. `dtype` : typage numérique
4. Formes (`shape`) et dimensions
5. Indexation et slicing
6. Indexation avancée : masks et fancy
7. Reshape, transpose, stack
8. Opérations élément par élément
9. Broadcasting
10. Réductions et agrégations
11. Axis — la notion clé
12. Fonctions universelles (`ufuncs`)
13. Algèbre linéaire
14. Nombres aléatoires (`Generator`)
15. Performance : pourquoi NumPy est rapide
16. Synthèse
17. Exercices


---


## 1. Installation et import

NumPy s'installe avec `pip install numpy` (ou `uv add numpy`). Son import canonique est :


In [ ]:
import numpy as np
np.__version__

La quasi-totalité du monde scientifique Python importe NumPy comme `np`. **Ne jamais** faire `from numpy import *` — cela écrase `sum`, `min`, `max` de Python.


---


## 2. Créer un `ndarray`

Le type central de NumPy est le `ndarray` (N-dimensional array). On peut le construire à partir d'une séquence Python, ou avec des constructeurs dédiés.


In [ ]:
np.array([1, 2, 3, 4])

In [ ]:
np.array([[1, 2, 3], [4, 5, 6]])

### Constructeurs utiles


In [ ]:
np.zeros(5)

In [ ]:
np.ones((2, 3))

In [ ]:
np.full((2, 2), 7)

In [ ]:
np.arange(0, 10, 2)  # comme range mais renvoie un ndarray

In [ ]:
np.linspace(0, 1, 5)  # 5 points régulièrement espacés entre 0 et 1

In [ ]:
np.eye(3)  # matrice identité 3x3

In [ ]:
np.empty((2, 2))  # alloué mais non initialisé — plus rapide

---


## 3. `dtype` : le typage numérique

Contrairement à une `list` Python, tous les éléments d'un `ndarray` partagent le **même type** stocké de façon compacte. Le `dtype` détermine la taille en mémoire et les valeurs représentables.


In [ ]:
a = np.array([1, 2, 3])
a.dtype

In [ ]:
b = np.array([1.0, 2.0, 3.0])
b.dtype

In [ ]:
c = np.array([1, 2, 3], dtype=np.int8)
c.dtype, c.nbytes

In [ ]:
d = np.array([1, 2, 3], dtype=np.float32)
d.dtype, d.nbytes

### Pourquoi choisir un `dtype` plus petit ?

Pour économiser la mémoire sur de gros volumes. Un `float64` fait 8 octets, un `float32` en fait 4. Sur un milliard d'éléments, c'est 4 Go versus 8 Go.


In [ ]:
np.array([1, 2, 3]).astype(np.float64)

### Attention aux overflows silencieux


In [ ]:
x = np.array([127], dtype=np.int8)
x + 1  # overflow silencieux — donne -128

---


## 4. Formes (`shape`) et dimensions

Un `ndarray` a une forme (`shape`), un nombre d'axes (`ndim`) et un nombre total d'éléments (`size`).


In [ ]:
a = np.array([[1, 2, 3], [4, 5, 6]])
a.shape, a.ndim, a.size

In [ ]:
b = np.zeros((2, 3, 4))
b.shape, b.ndim, b.size

Pour un tableau 3D `(2, 3, 4)`, lisez : **2 plans**, chacun de **3 lignes** × **4 colonnes**.


---


## 5. Indexation et slicing

L'indexation ressemble à celle des listes, mais supporte **plusieurs dimensions** séparées par des virgules.


In [ ]:
a = np.arange(12).reshape(3, 4)
a

In [ ]:
a[0, 0]  # ligne 0, colonne 0

In [ ]:
a[1, 2]  # ligne 1, colonne 2

In [ ]:
a[0]     # toute la ligne 0

In [ ]:
a[:, 1]  # toute la colonne 1

In [ ]:
a[0:2, 1:3]  # sous-bloc

### Les vues : attention aux effets de bord

Un slice NumPy ne **copie pas** les données : il crée une **vue**. Modifier la vue modifie le tableau d'origine.


In [ ]:
a = np.arange(5)
v = a[1:4]
v[0] = 99
a  # a est modifié

In [ ]:
a = np.arange(5)
c = a[1:4].copy()
c[0] = 99
a  # a est intact

---


## 6. Indexation avancée : masks et fancy

### Boolean masks


In [ ]:
a = np.array([3, 1, 4, 1, 5, 9, 2, 6])
mask = a > 3
mask

In [ ]:
a[a > 3]

Extrêmement puissant pour filtrer. Combinable avec `&` et `|` (parenthèses obligatoires) :


In [ ]:
a[(a > 2) & (a < 6)]

### Fancy indexing (liste d'indices)


In [ ]:
a[[0, 2, 4]]  # extrait les éléments 0, 2, 4

### Affectation par masque


In [ ]:
a = np.array([3, 1, 4, 1, 5])
a[a > 3] = 0
a

### `np.where` : if-else vectorisé


In [ ]:
a = np.array([1, -2, 3, -4, 5])
np.where(a >= 0, a, -a)  # valeur absolue

---


## 7. Reshape, transpose, stack


In [ ]:
a = np.arange(12)
a.reshape(3, 4)

In [ ]:
a.reshape(2, -1)  # -1 = à calculer

In [ ]:
a.reshape(3, 4).T  # transposée

### Concaténer des tableaux


In [ ]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
np.concatenate([a, b])

In [ ]:
np.stack([a, b])       # empile, ajoute un axe

In [ ]:
np.stack([a, b], axis=1)

In [ ]:
np.vstack([a, b])      # empilage vertical

In [ ]:
np.hstack([a, b])      # empilage horizontal

---


## 8. Opérations élément par élément

Les opérateurs Python (`+`, `-`, `*`, `/`, `**`, ...) sont **vectorisés** : ils s'appliquent terme à terme.


In [ ]:
a = np.array([1, 2, 3, 4])
b = np.array([10, 20, 30, 40])
a + b

In [ ]:
a * b

In [ ]:
a ** 2

In [ ]:
np.sqrt(a)

**Zéro boucle Python**, zéro conversion. Tout le travail est fait en C.


---


## 9. Broadcasting

Le **broadcasting** autorise les opérations entre tableaux de formes différentes, à condition qu'elles soient **compatibles**. NumPy aligne automatiquement les axes en partant de la droite.


In [ ]:
a = np.array([[1, 2, 3], [4, 5, 6]])  # shape (2, 3)
b = np.array([10, 20, 30])          # shape (3,)
a + b

La ligne `b` est **répétée virtuellement** sur les deux lignes de `a`.

### Règle

Deux formes sont compatibles si, aligne par la droite, chaque paire de dimensions est soit **égale**, soit l'une vaut **1**.


In [ ]:
x = np.arange(3).reshape(3, 1)  # (3, 1)
y = np.arange(4)                 # (4,)
x + y  # shape résultante : (3, 4)

### Exemple pratique : centrer les colonnes


In [ ]:
X = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
X - X.mean(axis=0)

### Exemple pratique : produit extérieur


In [ ]:
a = np.array([1, 2, 3])
b = np.array([10, 20])
a[:, None] * b[None, :]

---


## 10. Réductions et agrégations


In [ ]:
a = np.array([1, 2, 3, 4, 5])
a.sum(), a.mean(), a.std(), a.min(), a.max()

In [ ]:
a.cumsum()

In [ ]:
a.argmin(), a.argmax()  # indices du min et du max

In [ ]:
a.any(), a.all()  # vrai si au moins un / tous non nuls

---


## 11. `axis` — la notion clé

Une réduction sur un tableau 2D peut se faire **par ligne** ou **par colonne** selon l'axe choisi. `axis=0` réduit **en descendant les lignes** → on obtient un résultat par colonne.


In [ ]:
M = np.array([[1, 2, 3], [4, 5, 6]])
M.sum()           # sur tout

In [ ]:
M.sum(axis=0)     # somme des colonnes → [5, 7, 9]

In [ ]:
M.sum(axis=1)     # somme des lignes → [6, 15]

**Astuce mnémonique** : `axis=0` signifie « axe des lignes supprimé » → il ne reste que les colonnes.


---


## 12. Fonctions universelles (`ufuncs`)

Les `ufuncs` sont des fonctions NumPy qui opèrent **élément par élément**, très rapides car implémentées en C.


In [ ]:
a = np.array([0, np.pi/2, np.pi])
np.sin(a)

In [ ]:
np.exp(np.array([0, 1, 2]))

In [ ]:
np.log(np.array([1, np.e, np.e**2]))

In [ ]:
np.maximum(np.array([1, 4, 2]), np.array([3, 2, 5]))  # max terme à terme

### Écrire une `ufunc` personnalisée ?

Utilisez `np.vectorize` (syntaxique, pas rapide) ou Numba/Cython pour les cas critiques. Dans 99 % des cas, une combinaison de `ufuncs` existantes suffit.


---


## 13. Algèbre linéaire (`numpy.linalg`)


In [ ]:
A = np.array([[1.0, 2.0], [3.0, 4.0]])
b = np.array([5.0, 6.0])

In [ ]:
A @ b  # produit matriciel (opérateur @)

In [ ]:
A.T @ A

In [ ]:
np.linalg.inv(A)

In [ ]:
np.linalg.det(A)

In [ ]:
np.linalg.eig(A)

### Résoudre un système linéaire `A x = b`


In [ ]:
np.linalg.solve(A, b)

**Ne jamais** faire `np.linalg.inv(A) @ b` pour résoudre `A x = b`. `solve` est plus rapide et numériquement plus stable.


---


## 14. Nombres aléatoires avec `Generator`

**Depuis NumPy 1.17**, utilisez l'API `Generator` plutôt que les anciennes fonctions `np.random.rand`. Elle est plus rapide et statistiquement meilleure.


In [ ]:
rng = np.random.default_rng(seed=42)
rng.random(5)

In [ ]:
rng.integers(low=0, high=10, size=(3, 3))

In [ ]:
rng.normal(loc=0.0, scale=1.0, size=5)

In [ ]:
rng.choice(['pierre', 'feuille', 'ciseaux'], size=10)

Le `seed` garantit la **reproductibilité**. Utilisez toujours un `Generator` dédié dans vos scripts plutôt que les fonctions globales.


---


## 15. Performance : pourquoi NumPy est rapide


In [ ]:
# Comparaison : somme de 1 million de carrés
N = 1_000_000
import time
t0 = time.perf_counter()
s = sum(i*i for i in range(N))
t_py = time.perf_counter() - t0
t0 = time.perf_counter()
a = np.arange(N)
s2 = (a * a).sum()
t_np = time.perf_counter() - t0
print('python :', round(t_py, 3), 's')
print('numpy  :', round(t_np, 3), 's')
print('ratio  :', round(t_py / t_np, 1), 'x')

NumPy gagne grâce à trois choses : stockage contigu en mémoire, opérations en C sans passage par l'interpréteur, et vectorisation SIMD du processeur.


---


## Synthèse

| Notion | À retenir |
|---|---|
| `ndarray` | tableau N-D de dtype uniforme, stocké contigu en mémoire |
| `dtype` | type numérique (int8, float32, float64...) — affecte la taille mémoire |
| `shape` | tuple des dimensions, `ndim` = nombre d'axes, `size` = nb total d'éléments |
| Slicing | renvoie une **vue**, pas une copie — utiliser `.copy()` pour isoler |
| Mask booléen | `a[a > 3]` filtre, combinable avec `&` et `|` (parenthèses) |
| Broadcasting | alignement automatique des formes depuis la droite |
| `axis` | axe **supprimé** par la réduction (axis=0 réduit les lignes, il reste les colonnes) |
| `@` | produit matriciel, pas `*` qui est terme à terme |
| `Generator` | API aléatoire moderne, reproductible via `seed` |


### Règles à retenir

- **Éviter les boucles Python** : chercher une version vectorisée d'abord.
- **Jamais `from numpy import *`** — écrase les builtins Python.
- **Un slice est une vue** — utiliser `.copy()` si on veut isoler.
- **`solve(A, b)`** plutôt que `inv(A) @ b` pour résoudre un système.
- **`default_rng`** plutôt que les fonctions globales `np.random.rand`.


---


## Exercices

Pour chaque exercice : écrivez une **fonction typée**, testez-la, puis comparez avec la correction. N'ouvrez pas la correction avant d'avoir vraiment cherché.


### Exercice 1 — Normalisation d'un tableau

Écrivez une fonction `normaliser(x)` qui reçoit un `ndarray` 1D de floats et renvoie le tableau **centré et réduit** (moyenne 0, écart-type 1).


In [ ]:
import numpy as np

def normaliser(x: np.ndarray) -> np.ndarray:
    ...

print(normaliser(np.array([1.0, 2.0, 3.0, 4.0, 5.0])))


<details>
<summary>📖 Voir la correction</summary>

```python
import numpy as np

def normaliser(x: np.ndarray) -> np.ndarray:
    return (x - x.mean()) / x.std()

print(normaliser(np.array([1.0, 2.0, 3.0, 4.0, 5.0])))
```

</details>


### Exercice 2 — Filtrer avec un masque

Écrivez une fonction `garder_positifs(x)` qui renvoie uniquement les valeurs strictement positives d'un tableau, en utilisant un masque booléen.


In [ ]:
import numpy as np

def garder_positifs(x: np.ndarray) -> np.ndarray:
    ...


<details>
<summary>📖 Voir la correction</summary>

```python
import numpy as np

def garder_positifs(x: np.ndarray) -> np.ndarray:
    return x[x > 0]

print(garder_positifs(np.array([-2.0, -1.0, 0.0, 3.5, 7.0])))
```

</details>


### Exercice 3 — Matrice de distances par broadcasting

Étant donnés deux tableaux 1D `x` et `y` de floats, construisez par **broadcasting** la matrice `D` telle que `D[i, j] = |x[i] - y[j]|`. Aucune boucle Python.


In [ ]:
import numpy as np

def distances(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    ...

print(distances(np.array([0.0, 1.0, 2.0]), np.array([0.0, 3.0])))


<details>
<summary>📖 Voir la correction</summary>

```python
import numpy as np

def distances(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    return np.abs(x[:, None] - y[None, :])

print(distances(np.array([0.0, 1.0, 2.0]), np.array([0.0, 3.0])))
```

</details>


### Exercice 4 — Moyenne mobile

Écrivez une fonction `moyenne_mobile(x, k)` qui calcule la moyenne glissante de fenêtre `k` sur le tableau `x`. Retournez un tableau de longueur `len(x) - k + 1`. Indice : `np.cumsum` peut aider.


In [ ]:
import numpy as np

def moyenne_mobile(x: np.ndarray, k: int) -> np.ndarray:
    ...


<details>
<summary>📖 Voir la correction</summary>

```python
import numpy as np

def moyenne_mobile(x: np.ndarray, k: int) -> np.ndarray:
    c = np.cumsum(np.insert(x, 0, 0.0))
    return (c[k:] - c[:-k]) / k

print(moyenne_mobile(np.array([1.0, 2.0, 3.0, 4.0, 5.0]), k=3))
```

</details>


### Exercice 5 — Résolution d'un système linéaire

Écrivez une fonction `resoudre(A, b)` qui résout `A x = b` proprement avec `np.linalg.solve`. La fonction doit lever `ValueError` si `A` n'est pas carrée.


In [ ]:
import numpy as np

def resoudre(A: np.ndarray, b: np.ndarray) -> np.ndarray:
    ...


<details>
<summary>📖 Voir la correction</summary>

```python
import numpy as np

def resoudre(A: np.ndarray, b: np.ndarray) -> np.ndarray:
    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError('A doit être carrée')
    return np.linalg.solve(A, b)

A = np.array([[3.0, 1.0], [1.0, 2.0]])
b = np.array([9.0, 8.0])
print(resoudre(A, b))
```

</details>


---


## Ressources externes

- [Documentation officielle NumPy](https://numpy.org/doc/stable/)
- [NumPy User Guide](https://numpy.org/doc/stable/user/index.html)
- [NumPy Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html)
- [From Python to NumPy (Nicolas Rougier)](https://www.labri.fr/perso/nrougier/from-python-to-numpy/)
